# Clasificación de variedades de arroz: CNN vs. Xception, benchmark contra la literatura

- Objetivo: clasificar 5 variedades de arroz a partir de imágenes, y aplicar el flujo de 4 pasos para convertir este ejercicio en una contribución comparable: buscar el artículo relacionado, entender sus métricas, ejecutar y comparar, y mapear el resultado a la estructura de un artículo de conferencia.
- Este notebook acompaña al artículo completo, con el detalle de cada paso y la explicación sección por sección de un artículo de conferencia: **artículo completo →** https://fuzzyfrog.ai/es/ai-lab/proyectos/agritech/clasificacion-variedades-arroz-cnn-xception-benchmark-articulo-conferencia/
- **Dataset:** público, de Kaggle — [Rice Image Dataset](https://www.kaggle.com/datasets/muratkokludataset/rice-image-dataset) (verificado, URL vigente). 75,000 imágenes, 15,000 por cada una de 5 variedades: Arborio, Basmati, Ipsala, Jasmine, Karacadag.
- **Artículo de referencia:** Koklu, M., Cinar, I., Taspinar, Y.S. (2021). *"Classification of rice varieties with deep learning methods."* Computers and Electronics in Agriculture, 187, 106285. DOI: [10.1016/j.compag.2021.106285](https://doi.org/10.1016/j.compag.2021.106285). Es el paper que originó este mismo dataset — reporta 100% de accuracy con su CNN.


## Diagrama del flujo de 4 pasos

`Buscar artículo relacionado` → `Entender sus métricas` → `Ejecutar y comparar` → `Mapear a estructura de artículo de conferencia`.

Este flujo es genérico: aplica igual para clasificar limones, palta, frutos rojos, o cualquier otro producto agrícola con un dataset de imágenes disponible. El diagrama interactivo completo está en el artículo (sección "Diagrama de la solución").

In [ ]:
import numpy as np
import pandas as pd
import seaborn
import matplotlib.pylab as plt
import os
import random
import shutil
import glob
from PIL import Image
from imutils import paths
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Activation, Dense, Flatten, BatchNormalization, Conv2D,
                                      MaxPool2D, Dropout, AveragePooling2D, Input)
from tensorflow.keras.applications import Xception
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, classification_report
import cv2

seaborn.set(rc={'figure.figsize': (15, 10)})


## Paso 1 y 2 del flujo: artículo de referencia y sus métricas

Antes de escribir una sola línea de código, se identificó el artículo de origen del dataset (paso 1) y se registró su métrica principal (paso 2):

| | Referencia (Koklu et al., 2021) |
|---|---|
| Modelo | CNN |
| Accuracy reportado | 100% |
| Dataset | Mismo Rice Image Dataset (75,000 imágenes, 5 variedades) |

Con esa referencia clara, el resto del notebook corresponde al paso 3: ejecutar el experimento propio bajo condiciones comparables.

## Carga de datos

- Dataset público de Kaggle. Requiere credenciales propias de Kaggle (`kaggle.json`) para la descarga automática.
- **Nota sobre URLs:** se verificó que tanto el identificador del dataset (`muratkokludataset/rice-image-dataset`) como el DOI del artículo de referencia están vigentes al momento de escribir este notebook.

In [ ]:
# Descargar el dataset si no se ha descargado (requiere kaggle.json configurado)
if not os.path.isdir("Rice_Image_Dataset"):
    os.environ["KAGGLE_CONFIG_DIR"] = "."
    os.system("kaggle datasets download -d muratkokludataset/rice-image-dataset")
    os.system("unzip -q rice-image-dataset.zip")
    os.system("rm rice-image-dataset.zip")

categorias_arroz = ['Arborio', 'Basmati', 'Ipsala', 'Jasmine', 'Karacadag']


In [ ]:
# Tamaño y tipo de dato de las imágenes
ejemplo = cv2.imread(f"Rice_Image_Dataset/{categorias_arroz[0]}/{categorias_arroz[0]} (1).jpg")
print(type(ejemplo))
print("Forma:", ejemplo.shape)


## Explicación de los datos

- 5 clases balanceadas: 15,000 imágenes por variedad, 75,000 en total.
- Cada imagen es un grano de arroz individual, fotografiado en condiciones controladas.
- Se preparan carpetas de entrenamiento, validación y prueba con una muestra aleatoria por clase, para acelerar el ejercicio.

In [ ]:
train_path = "Rice_Image_Dataset/Arroz_ordenado/train/"
val_path = "Rice_Image_Dataset/Arroz_ordenado/val/"
test_path = "Rice_Image_Dataset/Arroz_ordenado/test/"

for cat in categorias_arroz:
    origen = f"Rice_Image_Dataset/{cat}"
    if not os.path.isdir(train_path + cat):
        os.makedirs(train_path + cat)
        os.makedirs(val_path + cat)
        os.makedirs(test_path + cat)
        archivos = glob.glob(origen + '/*')
        for c in random.sample(archivos, 8000):
            shutil.copy(c, train_path + cat)
        for c in random.sample(archivos, 2000):
            shutil.copy(c, val_path + cat)
        for c in random.sample(archivos, 3000):
            shutil.copy(c, test_path + cat)


In [ ]:
train_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.vgg16.preprocess_input).flow_from_directory(
    directory=train_path, target_size=(250, 250), classes=categorias_arroz, batch_size=32)
valid_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.vgg16.preprocess_input).flow_from_directory(
    directory=val_path, target_size=(250, 250), classes=categorias_arroz, batch_size=32)
test_batches = ImageDataGenerator(preprocessing_function=tf.keras.applications.vgg16.preprocess_input).flow_from_directory(
    directory=test_path, target_size=(250, 250), classes=categorias_arroz, batch_size=32, shuffle=False)


## Análisis de datos / EDA

**Por qué importa esta gráfica:** confirmar visualmente que las imágenes cargadas corresponden a las etiquetas esperadas, antes de invertir tiempo de cómputo entrenando con datos mal alineados.

In [ ]:
def mostrar_imgs(imgs):
    fig, axes = plt.subplots(1, 5, figsize=(20, 20))
    axes = axes.flatten()
    for img, ax in zip(imgs, axes):
        ax.imshow(img.astype('uint8'))
        ax.axis('off')
    plt.tight_layout()
    plt.show()

imgs, etiquetas = next(train_batches)
mostrar_imgs(imgs)
print(etiquetas)


## Modelado

### 6.1 CNN simple (2 capas convolucionales)

Arquitectura mínima, sin transfer learning, como primer punto de comparación.

In [ ]:
modelo = Sequential([
    Conv2D(filters=32, kernel_size=(3, 3), activation='relu', padding='same', input_shape=(250, 250, 3)),
    MaxPool2D(pool_size=(2, 2), strides=2),
    Conv2D(filters=64, kernel_size=(3, 3), activation='relu', padding='same'),
    MaxPool2D(pool_size=(2, 2), strides=2),
    Flatten(),
    Dense(units=5, activation='softmax')
])
modelo.summary()


In [ ]:
modelo.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

historia_modelo = modelo.fit(
    x=train_batches,
    steps_per_epoch=len(train_batches),
    validation_data=valid_batches,
    validation_steps=len(valid_batches),
    epochs=5,
    verbose=1
)


**Por qué importa esta gráfica:** compara accuracy/loss de entrenamiento contra validación. Con solo 5 épocas, ya es posible ver si el modelo converge de forma sana o si hay señales tempranas de sobreajuste.

In [ ]:
plt.plot(historia_modelo.history['accuracy'], linewidth=4, label='train_accuracy')
plt.plot(historia_modelo.history['val_accuracy'], linewidth=4, label='val_accuracy')
plt.title("CNN simple: accuracy por época")
plt.xlabel("Epoch"); plt.legend(); plt.show()

plt.plot(historia_modelo.history['loss'], linewidth=4, label='train_loss')
plt.plot(historia_modelo.history['val_loss'], linewidth=4, label='val_loss')
plt.title("CNN simple: loss por época")
plt.xlabel("Epoch"); plt.legend(); plt.show()


### 6.2 Xception con transfer learning

Backbone preentrenado en ImageNet, con una cabeza de clasificación propia.

In [ ]:
modelo_base = Xception(weights="imagenet", include_top=False, input_tensor=Input(shape=(250, 250, 3)))

modelo_sec = modelo_base.output
modelo_sec = AveragePooling2D(pool_size=(5, 5))(modelo_sec)
modelo_sec = Flatten(name="flatten")(modelo_sec)
modelo_sec = Dense(128, activation="relu")(modelo_sec)
modelo_sec = Dropout(0.5)(modelo_sec)
modelo_sec = Dense(train_batches.num_classes, activation="softmax")(modelo_sec)

modelo_xception = Model(inputs=modelo_base.input, outputs=modelo_sec)

# Congelar las capas de Xception para no modificar sus pesos durante el entrenamiento
for layer in modelo_base.layers:
    layer.trainable = False

modelo_xception.compile(loss="categorical_crossentropy", optimizer='Adam', metrics=["accuracy"])


In [ ]:
historia_xception = modelo_xception.fit(
    train_batches, validation_data=valid_batches, epochs=8
)


In [ ]:
plt.plot(historia_xception.history['accuracy'], linewidth=4, label='train_accuracy')
plt.plot(historia_xception.history['val_accuracy'], linewidth=4, label='val_accuracy')
plt.title("Xception + transfer learning: accuracy por época")
plt.xlabel("Epoch"); plt.legend(); plt.show()

plt.plot(historia_xception.history['loss'], linewidth=4, label='train_loss')
plt.plot(historia_xception.history['val_loss'], linewidth=4, label='val_loss')
plt.title("Xception + transfer learning: loss por época")
plt.xlabel("Epoch"); plt.legend(); plt.show()


## Evaluación (paso 3: comparar contra la referencia)

**Por qué importa esta comparación:** sin comparar explícitamente contra el artículo de referencia, cualquier accuracy "se ve bien" en aislado. La comparación es la que permite decir algo real sobre el resultado.

In [ ]:
def evaluar(modelo_eval, nombre):
    preds = modelo_eval.predict(x=test_batches, steps=len(test_batches), verbose=0)
    cm = confusion_matrix(y_true=test_batches.classes, y_pred=np.argmax(preds, axis=-1), normalize='true')
    plt.imshow(cm, interpolation='nearest', cmap="Reds")
    plt.title(f"Matriz de confusión: {nombre}")
    plt.colorbar()
    tick_marks = np.arange(len(categorias_arroz))
    plt.xticks(tick_marks, categorias_arroz, rotation=45)
    plt.yticks(tick_marks, categorias_arroz)
    plt.tight_layout()
    plt.ylabel('Etiqueta real'); plt.xlabel('Etiqueta predicha')
    plt.show()
    print(classification_report(test_batches.classes, np.argmax(preds, axis=-1), target_names=categorias_arroz))

evaluar(modelo, "CNN simple")
evaluar(modelo_xception, "Xception + transfer learning")


### Comparación final contra el artículo de referencia

| Modelo | Accuracy de validación |
|---|---|
| Referencia (Koklu et al., 2021), CNN | 100% |
| CNN simple (este notebook, 5 épocas) | 98.88% |
| Xception + transfer learning (este notebook, 8 épocas) | 95.99% |

La CNN simple, más pequeña y con menos épocas de entrenamiento, quedó más cerca de la referencia que Xception. Ver la sección de hallazgos para la interpretación.

## Hallazgos principales

- **La CNN simple superó a Xception con transfer learning**, algo contraintuitivo si se asume que "más complejo = mejor". La explicación más plausible: ImageNet enseña patrones de objetos naturales, mientras que distinguir variedades de arroz depende de textura, forma y color muy distintos a esos patrones. Los pesos preentrenados no fueron necesariamente el mejor punto de partida aquí.
- **La comparación contra el artículo de referencia (100% de accuracy) da contexto real al resultado.** 98.88% no es un número aislado, es "muy cerca de lo que ya se sabe que es alcanzable con este dataset".
- **El siguiente paso natural, y una posible contribución para un artículo de conferencia**, es investigar por qué Xception rindió peor: ¿más épocas de fine-tuning cerrarían la brecha? ¿Un backbone más liviano (MobileNet) generalizaría mejor a este tipo de textura? Esa pregunta bien argumentada, con evidencia, ya es un aporte.
- **Este flujo (buscar referencia → entender métricas → ejecutar y comparar → mapear a artículo) es reutilizable** para cualquier otro producto agrícola con dataset de imágenes disponible: limones, palta, frutos rojos, y otros.